# 🎯 Module 00: Setup and Environment

> **Course:** Generative AI — From Scratch to GenAI Engineer  
> **Stage:** Stage 1 — Foundations  
> **Module:** 00 — Setup and Environment  
> **Planned Date:** July 1, 2026  
> **Level:** Beginner → Intermediate  
> **Previous:** —  
> **Next:** 01 — GenAI Fundamentals

---

Before we can build any GenAI application, we need a **clean, reproducible, and secure development environment**.

This module is *not* about Python syntax — it's about the **engineering discipline** that separates a hobby project from a professional GenAI codebase.

We'll cover:

- What a GenAI development environment is
- Why environments matter
- **Virtual environments**
- **Package management**
- **Project structure**
- **Notebooks vs Python modules**
- **Environment variables & secrets**
- **Logging**
- **CPU vs GPU vs hosted APIs**
- Best practices, common mistakes, and security checklist

## 📚 Learning Objectives

After completing this module, you should be able to:

1. Explain why isolated Python environments are useful
2. Create and activate a virtual environment
3. Install and manage Python packages
4. Organize notebooks and reusable Python modules
5. Configure environment variables and API keys safely
6. Create a suitable `.gitignore`
7. Track changes with Git
8. Record dependencies for reproducible installations
9. Add basic logging and troubleshoot common errors
10. Explain when CPU, GPU, or a hosted model API is appropriate
11. Prepare a clean foundation for the remaining GenAI modules

---
## 🧠 1. What Is a GenAI Development Environment?

A **development environment** is the collection of software, libraries, configuration, and tools required to build and run an application.

For **Generative AI**, that typically includes:

- **Python** for application logic
- A **virtual environment** to isolate dependencies
- **Provider SDKs** to communicate with language models
- **Environment variables** to configure API credentials
- **Git** to track changes
- **Jupyter notebooks** for experimentation
- **Python modules** for reusable application code
- **Testing & logging tools** for reliability

### Example application flow

```text
Developer
    |
    v
Python Application
    |
    v
LLM Provider SDK
    |
    v
Model Provider API
    |
    v
Generated Response
```

> When you use a hosted API, inference usually happens on the **provider's infrastructure** — not on your computer.

---
## 🎯 2. Why Does This Module Matter?

A poorly configured environment causes real problems:

- Packages conflicting with one another
- "Works on my machine" bugs
- API keys accidentally pushed to GitHub
- Different dependency versions → different behavior
- Notebooks that can't be reproduced
- Hard-to-diagnose errors

> A professional AI engineer builds an environment that **another developer can replicate from documented instructions**.

This module is about **engineering setup** — not repeating Python fundamentals.

---
## ⚙️ 0. Setup & Imports

We start with the standard library — no external installs required for this section.

In [ ]:
import sys
import os
import platform
import logging
from pathlib import Path

print('✅ Python imports ready.')
print(f'Python version : {sys.version.split()[0]}')
print(f'Executable     : {sys.executable}')
print(f'Platform       : {platform.platform()}')

---
## 🧪 3. Python & Virtual Environments

A **virtual environment** is an isolated location for the packages used by a particular project.

### Why?

Suppose **Project A** needs `library==1.0` and **Project B** needs `library==2.0`. Installing everything globally creates **conflicts**.

A virtual environment keeps them separate.

### Create & activate

```bash
mkdir genai-learning
cd genai-learning
python -m venv .venv
```

**Windows PowerShell:**
```powershell
.venv\Scripts\Activate.ps1
```

**macOS / Linux:**
```bash
source .venv/bin/activate
```

### Verify

```bash
python --version
python -c "import sys; print(sys.executable)"
```

When active, the interpreter should point to the project's `.venv` directory.

### Leave

```bash
deactivate
```

> ⚠️ A virtual environment isolates **Python packages**. It is **not** an OS sandbox or security boundary.

In [ ]:
# Detect whether the current interpreter is inside a virtual environment
in_venv = (
    hasattr(sys, 'real_prefix')                    # old virtualenv
    or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)  # venv / virtualenv
)

print('Running inside a virtual environment:', in_venv)
print('sys.prefix      :', sys.prefix)
print('sys.base_prefix :', getattr(sys, 'base_prefix', sys.prefix))

if not in_venv:
    print('\n⚠️ You are NOT in a virtual environment.')
    print('   Create one with: python -m venv .venv')
    print('   Activate it before installing packages.')
else:
    print('\n✅ Good — you are inside an isolated environment.')

---
## 📦 4. Package Management

Python packages provide reusable functionality — like **provider SDKs** that talk to a model API.

### Install

```bash
python -m pip install --upgrade pip
python -m pip install python-dotenv pydantic pytest
```

### Modern alternative: `uv`

Some teams prefer [`uv`](https://github.com/astral-sh/uv) as a faster, modern package manager.
**Pick one workflow** and document it.

### Record dependencies

```bash
python -m pip freeze > requirements.txt
```

### Reinstall elsewhere

```bash
python -m pip install -r requirements.txt
```

> `pip freeze` records **installed versions** — useful, but not a complete dependency-management strategy for large projects. For those, use an explicit dependency file and a **lockfile** (e.g., Poetry, uv, pip-tools).

In [ ]:
# Check installed versions of a few core packages (may fail if not installed yet)
for pkg in ['pip', 'pydantic', 'dotenv', 'pytest']:
    try:
        module = __import__(pkg)
        version = getattr(module, '__version__', 'unknown')
        print(f'✅ {pkg:10s} → {version}')
    except ImportError:
        print(f'❌ {pkg:10s} → not installed')

---
## 🏗️ 5. Project Structure

Keep **experiments** separate from **reusable code**.

```text
00-Setup-and-Environment/
|
├── README.md
├── requirements.txt
├── .env.example
├── .gitignore
|
├── notebooks/
|   └── 01-environment-check.ipynb
|
├── src/
|   └── genai_setup/
|       └── __init__.py
|
├── examples/
|   └── environment_check.py
|
└── tests/
    └── test_environment.py
```

> This is a **teaching example** — not every file must exist on day one.

In [ ]:
# Inspect the current working directory
cwd = Path.cwd()
print('Current working directory:', cwd)
print()
print('Top-level items:')
for item in sorted(cwd.iterdir()):
    kind = 'DIR ' if item.is_dir() else 'FILE'
    print(f'  [{kind}] {item.name}')

---
## 📓 6. Notebooks vs Python Modules

| Feature | Jupyter notebook | Python module |
| --- | --- | --- |
| Best use | Experiments and demonstrations | Reusable application logic |
| Execution | Cell by cell | Script, import, or entry point |
| Strength | Immediate feedback & explanation | Testing, reuse, maintainability |
| Limitation | Hidden execution state can confuse | Less convenient for exploration |

> **Notebooks** to understand concepts.  
> **Modules** to build maintainable applications.

---
## 🔐 7. Environment Variables & Secrets

> **Never hardcode real API keys in source code.**

### `.env.example` (safe to commit)

```dotenv
# Replace placeholders with your local configuration.
OPENAI_API_KEY=your_api_key_here
MODEL_NAME=your_chosen_model
```

### `.gitignore` (keep secrets out of Git)

```gitignore
.venv/
__pycache__/
*.py[cod]
.env
.ipynb_checkpoints/
.pytest_cache/
.ruff_cache/
.DS_Store
```

> `.env.example` can be committed — it contains **placeholders**, not secrets.

### ⚠️ For production

Use your **hosting platform's secret manager** instead of relying on a local `.env` file.

In [ ]:
# Demo: load .env safely, if python-dotenv is installed
try:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv('OPENAI_API_KEY')

    if api_key:
        # Show only a masked preview — NEVER the full key
        masked = api_key[:4] + '...' + api_key[-4:] if len(api_key) > 8 else '(too short)'
        print(f'OPENAI_API_KEY loaded: {masked}')
    else:
        print('OPENAI_API_KEY not set — this is fine for now.')
        print('Create a .env file from .env.example when you have a real key.')
except ImportError:
    print('python-dotenv not installed — pip install python-dotenv')

---
## 📋 8. Basic Logging

Logging helps developers understand what an application is doing **in real time** — without littering code with `print()`.

### Best practice

- Log **operational metadata** (timestamps, statuses, paths)
- **Never** log API keys, passwords, private prompts, or sensitive user data

In [ ]:
# Configure a simple logger
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s',
)

logging.info('GenAI environment check started')
logging.info('Python version: %s', sys.version.split()[0])
logging.info('Working directory: %s', Path.cwd())
logging.info('Environment check completed')

---
## 🔄 9. How a GenAI Environment Works Internally

A hosted-LLM application typically follows this lifecycle:

```text
1. Configure the application
          |
          v
2. Load settings and secrets
          |
          v
3. Validate configuration
          |
          v
4. Execute Python application
          |
          v
5. Send request through provider SDK
          |
          v
6. Receive and process the response
          |
          v
7. Log safe operational information
```

The SDK handles **serialization and communication**. Your application is still responsible for:

- Configuration
- Validation
- Error handling
- Safe processing of responses

---
## 🖥️ 10. CPU, GPU, and Hosted Inference

| Approach | What happens | Typical use |
| --- | --- | --- |
| **CPU** | Computation on the processor | Setup, preprocessing, small models |
| **GPU** | Compatible workloads on a GPU | Training, accelerated local inference |
| **Hosted API** | Provider runs the model remotely | Learning APIs, building apps without hardware |

> 🎯 **You do NOT need a powerful GPU to begin learning GenAI** — hosted APIs are enough.

Local inference can reduce dependence on external APIs, but adds hardware, memory, licensing, deployment, and performance considerations.

---
## ✅ 11. Best Practices

- Use **one virtual environment per project**
- Document the supported **Python version**
- Install only the packages you need
- Keep **secrets out of source control**
- Use `.env.example` to document required config
- Record dependencies and verify **fresh installs**
- Use **Git commits** to track meaningful changes
- Separate **exploratory notebooks** from **reusable code**
- Use **logging** instead of scattered `print()` statements
- Test code in a **clean environment** before sharing
- Document any **paid API / hardware / service** requirements

---
## ⚠️ 12. Common Mistakes

| Mistake | Solution |
| --- | --- |
| Installing packages into the wrong interpreter | Check `sys.executable` and activate the correct environment |
| Committing `.env` | Add it to `.gitignore`; rotate exposed credentials |
| Assuming a GPU is mandatory | Begin with hosted APIs or CPU-friendly examples |
| Relying on notebook execution order | Restart the kernel and run cells top-to-bottom |
| Installing every AI framework | Install only what the current exercise requires |
| Ignoring dependency versions | Document and test the environment |
| Logging sensitive information | Log operational metadata only |

> ⚠️ If a secret is accidentally pushed to a **public** repo:
> 1. Removing the file is **not enough**.
> 2. **Revoke or rotate** the exposed credential **immediately**.

---
## 🔒 13. Security & Reliability Checklist

Before moving to the next module, verify:

- [ ] The virtual environment works
- [ ] Required packages install successfully
- [ ] The Python version is documented
- [ ] `.env` is excluded from Git
- [ ] `.env.example` contains placeholders only
- [ ] No real API keys exist in committed files
- [ ] The example script runs successfully
- [ ] Git changes can be reviewed before committing
- [ ] Dependency installation is reproducible
- [ ] Any external service requirements are documented

---
## 🏋️ 14. Practical Exercises

### 🟢 Beginner — Create Your Environment
Create a virtual environment, install **Pydantic** and **Pytest**, and write a script that prints:
- Python version
- Project directory

**Acceptance criteria:** script runs inside the venv, output documented.

---

### 🟡 Intermediate — Safe Configuration
Create:
- `.env.example`
- `.gitignore`
- A configuration-loading example using placeholders

Verify that **missing required configuration raises a clear error**.

**Acceptance criteria:** no secrets hardcoded; missing config is handled explicitly.

---

### 🔴 Challenge — Reproducible GenAI Starter
Build a small project containing:
- A Python module
- A notebook
- Logging
- Dependency documentation
- A basic test

Then test it from a **fresh virtual environment**.

**Acceptance criteria:** another developer can follow your README, install deps, run the example, and run the tests.

---
## 🧪 15. Hands-On Demo — An Environment Health Check

The cell below combines everything we've covered: Python version, virtual environment detection, working directory, and safe logging.

In [ ]:
# A complete environment health check — copy this pattern into your own projects
import sys, os, platform, logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s',
)

def check_environment():
    logging.info('GenAI environment check started')

    # 1. Python
    logging.info('Python version: %s', sys.version.split()[0])

    # 2. Virtual environment
    in_venv = (
        hasattr(sys, 'real_prefix') or
        (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
    )
    logging.info('Inside virtual environment: %s', in_venv)

    # 3. Working directory
    logging.info('Working directory: %s', Path.cwd())

    # 4. Platform
    logging.info('Platform: %s', platform.platform())

    # 5. Required env vars (presence only — never the value)
    for var in ['OPENAI_API_KEY', 'MODEL_NAME']:
        present = bool(os.getenv(var))
        logging.info('Env var %s present: %s', var, present)

    logging.info('Environment check completed')
    return True

check_environment()

---
## 🎤 16. Interview Questions

**Q1. Why should you use a virtual environment?**  
It isolates project dependencies and reduces conflicts between projects.

**Q2. What is the difference between a package and a virtual environment?**  
A **package** provides functionality. A **virtual environment** isolates an interpreter and its installed packages.

**Q3. Why should API keys not be committed to GitHub?**  
Anyone with access to the committed secret may misuse it — causing security incidents or unexpected charges.

**Q4. When do you need a GPU for GenAI?**  
A GPU is useful for **local training and inference**. Hosted APIs let you use remote infrastructure without owning a GPU.

**Q5. Why are notebooks not always suitable for production application logic?**  
They can depend on **hidden execution state** and are harder to structure and test consistently than Python modules.

**Q6. What is the purpose of a dependency lockfile?**  
It records **resolved dependency versions** so compatible environments can be recreated consistently.

**Q7. What is the difference between logging and testing?**  
**Logging** records events during execution. **Testing** checks whether code behaves as expected.

---
## 🏁 17. Key Takeaways

- A clean environment makes GenAI projects easier to **develop and reproduce**
- **Virtual environments** prevent many dependency conflicts
- **API credentials** must be protected
- **Hosted APIs** are sufficient for starting GenAI development
- **Notebooks** support experimentation; **modules** support reusable engineering
- **Logging**, **testing**, **dependency management**, and **Git** are part of professional development

### 🧭 Golden Rule

> **If another developer cannot reproduce your environment from your README, your project is not done.**

---

### 🔗 What's Next?

**Module 01 — GenAI Fundamentals**

You'll learn what Generative AI is, how it differs from traditional Machine Learning, and how text, image, audio, video, and multimodal models fit into real-world applications.

> The objective is not merely to run code — it's to **understand what you're building and why**.